# Telecom τ²-bench ↔ p2m Correlation Report

This notebook reads the JSON results produced by `run_comparison.py` and generates
a detailed analysis report with tables, charts, and statistical interpretation.

**Prerequisites**: Run the pipeline first:
```bash
python run_comparison.py --preset quick   # or mini / full
```

The notebook expects these files in `results/`:
- `correlation_results.json` — combined results with correlations
- `tau2_rewards.json` — intermediate tau2 rewards
- `p2m_scores.json` — intermediate p2m scores

In [ ]:
import json
from pathlib import Path
from collections import Counter

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy.stats import spearmanr

sns.set_theme(style="whitegrid", font_scale=1.1)
pd.set_option("display.precision", 4)

RESULTS_DIR = Path("results")
SIM_DIR = Path("data/simulations")

In [ ]:
# ── Load results ──────────────────────────────────────────────────
results_path = RESULTS_DIR / "correlation_results.json"
if not results_path.exists():
    raise FileNotFoundError(
        f"{results_path} not found. Run the pipeline first:\n"
        "  python run_comparison.py --preset quick"
    )

results = json.loads(results_path.read_text())
tau2_rewards = results["tau2_rewards"]
tau2_samples = results.get("tau2_sample_sizes", {})
p2m_scores = results["p2m_scores"]
correlations = results["correlations"]
models = results.get("models_compared", sorted(set(tau2_rewards) & set(p2m_scores)))

def slug(model: str) -> str:
    """Short display name."""
    return model.split("/")[-1] if "/" in model else model

print(f"Models compared: {len(models)}")
for m in models:
    n = tau2_samples.get(m, "?")
    print(f"  {slug(m):25s}  tau2 n={n}")

---
## 1. Model Scores Overview

Side-by-side comparison of tau2 mean reward and p2m overall score for each model.

In [ ]:
# ── Build overview table ──────────────────────────────────────────
rows = []
for m in models:
    row = {
        "Model": slug(m),
        "τ² Mean Reward": tau2_rewards.get(m, float("nan")),
        "τ² Simulations": tau2_samples.get(m, 0),
        "p2m Overall": p2m_scores.get(m, {}).get("_overall", float("nan")),
    }
    # Add individual p2m dimensions
    for dim, val in sorted(p2m_scores.get(m, {}).items()):
        if dim != "_overall":
            row[f"p2m {dim}"] = val
    rows.append(row)

df_overview = pd.DataFrame(rows).set_index("Model")
df_overview.style.format("{:.4f}", subset=[c for c in df_overview.columns if c != "τ² Simulations"]).background_gradient(
    cmap="RdYlGn", subset=["τ² Mean Reward", "p2m Overall"]
)

In [ ]:
# ── Bar chart: tau2 vs p2m ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
x = range(len(models))
labels = [slug(m) for m in models]
tau2_vals = [tau2_rewards.get(m, 0) for m in models]
p2m_vals = [p2m_scores.get(m, {}).get("_overall", 0) for m in models]

width = 0.35
bars1 = ax.bar([i - width/2 for i in x], tau2_vals, width, label="τ² Mean Reward", color="#4C72B0")
bars2 = ax.bar([i + width/2 for i in x], p2m_vals, width, label="p2m Overall Score", color="#55A868")

ax.set_ylabel("Score")
ax.set_title("Model Performance: τ²-bench vs p2m")
ax.set_xticks(list(x))
ax.set_xticklabels(labels, rotation=30, ha="right")
ax.legend()
ax.set_ylim(0, 1.05)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0))

# Value labels
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f"{bar.get_height():.2f}", ha="center", va="bottom", fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f"{bar.get_height():.2f}", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "model_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 2. Spearman Rank Correlation

Spearman ρ measures monotonic rank agreement between tau2 rewards and p2m scores.
- **ρ = 1.0**: perfect positive rank agreement
- **ρ = 0.0**: no monotonic relationship
- **ρ = -1.0**: perfect inverse rank agreement
- **p < 0.05**: statistically significant at 95% confidence

In [ ]:
# ── Correlation table ─────────────────────────────────────────────
corr_rows = []
for dim in sorted(correlations):
    c = correlations[dim]
    corr_rows.append({
        "Dimension": dim,
        "Spearman ρ": c["rho"],
        "p-value": c["pval"],
        "n (models)": c["n"],
        "Significant (p<0.05)": "✓" if c["pval"] < 0.05 else "",
    })

df_corr = pd.DataFrame(corr_rows).set_index("Dimension")
df_corr.style.format({"Spearman ρ": "{:.4f}", "p-value": "{:.4f}"}).applymap(
    lambda v: "background-color: #d4edda" if v == "✓" else "",
    subset=["Significant (p<0.05)"]
)

In [ ]:
# ── Correlation heatmap ───────────────────────────────────────────
dims = sorted(d for d in correlations if d != "_overall")
if dims:
    rho_vals = [correlations[d]["rho"] for d in dims]
    pval_vals = [correlations[d]["pval"] for d in dims]

    fig, axes = plt.subplots(1, 2, figsize=(14, max(3, len(dims) * 0.7)))

    # Rho heatmap
    rho_df = pd.DataFrame({"Spearman ρ": rho_vals}, index=dims)
    sns.heatmap(rho_df, annot=True, fmt=".3f", cmap="RdBu_r", center=0,
                vmin=-1, vmax=1, ax=axes[0], cbar_kws={"shrink": 0.8})
    axes[0].set_title("Spearman ρ by Dimension")

    # p-value heatmap
    pval_df = pd.DataFrame({"p-value": pval_vals}, index=dims)
    sns.heatmap(pval_df, annot=True, fmt=".3f", cmap="YlOrRd_r",
                vmin=0, vmax=1, ax=axes[1], cbar_kws={"shrink": 0.8})
    axes[1].set_title("p-values (< 0.05 = significant)")

    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "correlation_heatmap.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("Only _overall correlation available; skipping dimension heatmap.")

---
## 3. p2m Dimension Breakdown

Violation rates per dimension per model. Lower = better agent behavior.

In [ ]:
# ── Grouped bar chart: p2m dimensions ─────────────────────────────
all_dims = sorted({d for scores in p2m_scores.values() for d in scores if d != "_overall"})

fig, ax = plt.subplots(figsize=(12, 5))
x = range(len(all_dims))
n_models = len(models)
total_width = 0.8
bar_width = total_width / n_models
colors = sns.color_palette("Set2", n_models)

for i, m in enumerate(models):
    vals = [p2m_scores.get(m, {}).get(d, 0) for d in all_dims]
    offset = (i - n_models / 2 + 0.5) * bar_width
    ax.bar([xi + offset for xi in x], vals, bar_width, label=slug(m), color=colors[i])

ax.set_ylabel("Violation Rate")
ax.set_title("p2m Violation Rates by Dimension")
ax.set_xticks(list(x))
ax.set_xticklabels([d.replace("_", " ").title() for d in all_dims], rotation=30, ha="right")
ax.legend(title="Model")
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0))

plt.tight_layout()
plt.savefig(RESULTS_DIR / "p2m_dimensions.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 4. τ²-bench Simulation Details

Per-model breakdown of tau2 simulations: reward distribution, completion rates,
cost, and per-task performance.

In [ ]:
# ── Load raw simulation data ──────────────────────────────────────
sim_data = {}
for m in models:
    path = SIM_DIR / f"telecom_{slug(m)}.json"
    if path.exists():
        data = json.loads(path.read_text())
        sim_data[m] = data["simulations"]

if sim_data:
    detail_rows = []
    for m, sims in sim_data.items():
        rewards = [s["reward_info"]["reward"] for s in sims]
        costs = [s.get("agent_cost", 0) + s.get("user_cost", 0) for s in sims]
        durations = [s.get("duration", 0) for s in sims]
        terms = Counter(s.get("termination_reason", "unknown") for s in sims)
        detail_rows.append({
            "Model": slug(m),
            "Simulations": len(sims),
            "Mean Reward": sum(rewards) / len(rewards),
            "Success Rate": sum(1 for r in rewards if r == 1.0) / len(rewards),
            "Failure Rate": sum(1 for r in rewards if r == 0.0) / len(rewards),
            "Total Cost ($)": sum(costs),
            "Avg Cost/Sim ($)": sum(costs) / len(costs),
            "Avg Duration (s)": sum(durations) / len(durations) if durations else 0,
            "Termination": ", ".join(f"{k}: {v}" for k, v in terms.most_common()),
        })

    df_detail = pd.DataFrame(detail_rows).set_index("Model")
    display(df_detail.style.format({
        "Mean Reward": "{:.4f}",
        "Success Rate": "{:.1%}",
        "Failure Rate": "{:.1%}",
        "Total Cost ($)": "${:.4f}",
        "Avg Cost/Sim ($)": "${:.6f}",
        "Avg Duration (s)": "{:.1f}",
    }).background_gradient(cmap="RdYlGn", subset=["Mean Reward", "Success Rate"]))
else:
    print("No simulation files found in", SIM_DIR)

In [ ]:
# ── Reward distribution histograms ────────────────────────────────
if sim_data:
    n = len(sim_data)
    fig, axes = plt.subplots(1, n, figsize=(5 * n, 4), sharey=True)
    if n == 1:
        axes = [axes]

    for ax, (m, sims) in zip(axes, sim_data.items()):
        rewards = [s["reward_info"]["reward"] for s in sims]
        ax.hist(rewards, bins=20, color="#4C72B0", edgecolor="white", alpha=0.8)
        ax.axvline(sum(rewards)/len(rewards), color="red", linestyle="--",
                   label=f"mean={sum(rewards)/len(rewards):.3f}")
        ax.set_title(slug(m))
        ax.set_xlabel("Reward")
        ax.legend(fontsize=9)

    axes[0].set_ylabel("Count")
    fig.suptitle("τ²-bench Reward Distribution", fontsize=14)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "reward_distribution.png", dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
# ── Per-task reward comparison ────────────────────────────────────
if len(sim_data) >= 2:
    # Gather per-task mean reward across models
    task_rewards: dict[str, dict[str, float]] = {}  # task_id -> {model: reward}
    for m, sims in sim_data.items():
        by_task: dict[str, list[float]] = {}
        for s in sims:
            tid = s["task_id"]
            by_task.setdefault(tid, []).append(s["reward_info"]["reward"])
        for tid, vals in by_task.items():
            task_rewards.setdefault(tid, {})[slug(m)] = sum(vals) / len(vals)

    # Find tasks present in all models
    model_slugs = [slug(m) for m in sim_data]
    common_tasks = [t for t, mr in task_rewards.items() if all(s in mr for s in model_slugs)]

    if common_tasks:
        task_df = pd.DataFrame([
            {"task_id": t, **task_rewards[t]} for t in common_tasks
        ]).set_index("task_id").sort_index()

        print(f"\nTasks with data in all {len(model_slugs)} models: {len(common_tasks)}")

        # Show tasks where models disagree most
        task_df["spread"] = task_df.max(axis=1) - task_df.min(axis=1)
        top_disagreements = task_df.nlargest(15, "spread").drop(columns=["spread"])
        print("\nTop 15 tasks with largest model disagreement:")
        display(top_disagreements.style.background_gradient(cmap="RdYlGn", axis=None))
    else:
        print("No tasks found across all models.")

---
## 5. Scatter Plots: τ² vs p2m per Dimension

Each point is a model. The fitted line (if any) shows the Spearman trend.

In [ ]:
# ── Scatter plots ─────────────────────────────────────────────────
dims_to_plot = ["_overall"] + sorted(d for d in correlations if d != "_overall")
n_plots = len(dims_to_plot)
ncols = min(3, n_plots)
nrows = (n_plots + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4.5 * nrows))
axes_flat = [axes] if n_plots == 1 else axes.flat

for idx, dim in enumerate(dims_to_plot):
    ax = axes_flat[idx]
    tau2_v, p2m_v, labels = [], [], []
    for m in models:
        if dim in p2m_scores.get(m, {}):
            t = tau2_rewards[m]
            p = p2m_scores[m][dim]
            if dim != "_overall":
                p = 1.0 - p  # convert violation rate to success rate
            tau2_v.append(t)
            p2m_v.append(p)
            labels.append(slug(m))

    ax.scatter(tau2_v, p2m_v, s=100, zorder=5)
    for xi, yi, label in zip(tau2_v, p2m_v, labels):
        ax.annotate(label, (xi, yi), textcoords="offset points",
                    xytext=(5, 5), fontsize=8)

    c = correlations.get(dim, {})
    rho = c.get("rho", float("nan"))
    pval = c.get("pval", float("nan"))
    title = dim.replace("_", " ").title()
    sig = " *" if pval < 0.05 else ""
    ax.set_title(f"{title}\nρ={rho:.3f}, p={pval:.3f}{sig}")
    ax.set_xlabel("τ² Mean Reward")
    ylabel = "p2m Overall" if dim == "_overall" else f"p2m {title} (success rate)"
    ax.set_ylabel(ylabel)

# Hide unused subplots
for idx in range(n_plots, len(list(axes_flat))):
    axes_flat[idx].set_visible(False)

fig.suptitle("τ² Reward vs p2m Score — Scatter by Dimension", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "scatter_plots.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 6. Data Quality & Reliability

Checks for data completeness and flags potential reliability concerns.

In [ ]:
# ── Data quality checks ───────────────────────────────────────────
print("Data Quality Summary")
print("=" * 50)

# 1. Sample sizes
print("\n1. τ²-bench Sample Sizes:")
for m in models:
    n = tau2_samples.get(m, 0)
    status = "✓" if n >= 100 else "⚠ LOW" if n >= 30 else "✗ VERY LOW"
    print(f"   {slug(m):25s}  n={n:>5}  {status}")

# 2. Simulation completeness
print("\n2. Simulation Completeness:")
if sim_data:
    # Get expected task count from first model with data
    for m, sims in sim_data.items():
        data = json.loads((SIM_DIR / f"telecom_{slug(m)}.json").read_text())
        n_tasks = len(data.get("tasks", []))
        n_trials = data.get("info", {}).get("num_trials", 1)
        expected = n_tasks * n_trials
        actual = len(sims)
        pct = actual / expected * 100 if expected else 0
        status = "✓" if pct >= 95 else "⚠ INCOMPLETE" if pct >= 50 else "✗ VERY INCOMPLETE"
        print(f"   {slug(m):25s}  {actual:>4}/{expected:<4} ({pct:.0f}%)  {status}")

# 3. Correlation reliability
n_models = len(models)
print(f"\n3. Correlation Reliability:")
print(f"   Models compared: {n_models}")
if n_models < 5:
    print(f"   ⚠ With only {n_models} models, Spearman ρ has very low statistical power.")
    print(f"     Minimum for p<0.05 with perfect correlation: n=5.")
    print(f"     Add more models (--preset full) for meaningful statistics.")
elif n_models < 8:
    print(f"   ⚠ Moderate power. Consider adding more models for robustness.")
else:
    print(f"   ✓ Good sample size for rank correlation.")

# 4. Significant correlations
sig_dims = [d for d, c in correlations.items() if c["pval"] < 0.05]
print(f"\n4. Significant Correlations (p<0.05): {len(sig_dims)}/{len(correlations)}")
if sig_dims:
    for d in sig_dims:
        c = correlations[d]
        print(f"   ✓ {d}: ρ={c['rho']:.4f} (p={c['pval']:.4f})")
else:
    print("   No significant correlations found.")
    if n_models < 5:
        print("   (Expected — too few models for statistical significance.)")

---
## 7. Summary & Interpretation

In [ ]:
# ── Auto-generated summary ────────────────────────────────────────
from IPython.display import Markdown

overall_rho = correlations.get("_overall", {}).get("rho", float("nan"))
overall_pval = correlations.get("_overall", {}).get("pval", float("nan"))

# Rank models
tau2_rank = sorted(models, key=lambda m: tau2_rewards.get(m, 0), reverse=True)
p2m_rank = sorted(models, key=lambda m: p2m_scores.get(m, {}).get("_overall", 0), reverse=True)

lines = [
    "### Key Findings\n",
    f"- **{len(models)} models** compared across {len(correlations)} dimensions.",
    f"- **Overall Spearman ρ = {overall_rho:.4f}** (p = {overall_pval:.4f}).",
]

if overall_pval < 0.05:
    direction = "positive" if overall_rho > 0 else "negative"
    lines.append(f"  - Statistically significant {direction} correlation between τ² and p2m.")
else:
    lines.append(f"  - Not statistically significant (p ≥ 0.05). "
                 f"More models needed for conclusive results.")

lines.append(f"\n### Model Rankings\n")
lines.append("| Rank | τ²-bench (reward) | p2m (overall) |")
lines.append("|------|------------------|---------------|")
for i in range(len(models)):
    t = slug(tau2_rank[i]) if i < len(tau2_rank) else "-"
    p = slug(p2m_rank[i]) if i < len(p2m_rank) else "-"
    lines.append(f"| {i+1} | {t} | {p} |")

if sig_dims:
    lines.append(f"\n### Significant Dimensions\n")
    for d in sig_dims:
        c = correlations[d]
        lines.append(f"- **{d}**: ρ={c['rho']:.4f} (p={c['pval']:.4f})")

low_sample = [m for m in models if tau2_samples.get(m, 0) < 50]
if low_sample:
    lines.append(f"\n### Caveats\n")
    lines.append(f"- Models with low τ² sample count (<50): "
                 f"{', '.join(slug(m) for m in low_sample)}")
    lines.append("- Re-run with `--stages tau2 --models ...` to improve coverage.")

if n_models < 5:
    if "### Caveats\n" not in lines:
        lines.append(f"\n### Caveats\n")
    lines.append(f"- Only {n_models} models compared. Spearman correlation needs ≥5 models "
                 f"for meaningful significance testing.")

Markdown("\n".join(lines))

In [ ]:
# ── Save charts list ──────────────────────────────────────────────
charts = list(RESULTS_DIR.glob("*.png"))
print(f"\n📊 Charts saved to {RESULTS_DIR}/")
for c in sorted(charts):
    print(f"   {c.name}")